# Benchmarking & Profiling

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/03_benchmarking.ipynb)

This notebook covers:
1. Analytical FLOPs estimation (`count_flops`)
2. Wall-clock latency measurement (`LatencyTracker`)
3. Full benchmark suite (`BenchmarkSuite.default()`)
4. Visualization (`Visualizer`)

In [ ]:
!pip install -q git+https://github.com/winstonsmith1897/DantinoX.git#egg=dantinox[all]

In [ ]:
import dantinox as dx
from dantinox.benchmarking import BenchmarkSuite, BenchmarkConfig
from dantinox.profiling import LatencyTracker, count_flops
from flax import nnx
import jax
import jax.numpy as jnp

cfg      = dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                           vocab_size=2000, causal=True)
paradigm = dx.ARParadigm(cfg)
model    = paradigm.build_model(nnx.Rngs(0))
print(f'Parameters: {paradigm.num_parameters(model):,}')

## 1. Analytical FLOPs

In [ ]:
for seq_len in [64, 128, 256, 512]:
    flops = count_flops(cfg, seq_len=seq_len, batch_size=1)
    print(f'seq_len={seq_len:4d}  total={flops.total / 1e9:.2f} GFLOPs')

## 2. Wall-clock latency

In [ ]:
tracker = LatencyTracker()
x = jax.random.randint(jax.random.PRNGKey(0), (4, 256), 0, 2000)

# Warmup
for _ in range(5):
    _ = model(x)

# Measure
for _ in range(20):
    with tracker.measure(n_tokens=4 * 256):
        _ = model(x)

r = tracker.result()
print(r)

## 3. Full benchmark suite

In [ ]:
config = BenchmarkConfig(
    seq_lens=[64, 128, 256],
    batch_sizes=[1, 4, 16],
    n_warmup=3, n_measure=10,
    eval_batches=10,
)
report = BenchmarkSuite.default(config).run(paradigm, model, save_csv='/tmp/bench.csv')
print(report.summary())

## 4. Visualization

In [ ]:
import pandas as pd
from dantinox.visualization import Visualizer
import os

os.makedirs('/tmp/plots', exist_ok=True)
df    = pd.read_csv('/tmp/bench.csv')
paths = Visualizer().render(df, out_dir='/tmp/plots')

from IPython.display import Image, display
for name, path in paths.items():
    print(name)
    display(Image(str(path)))